# Fitting single crystal parameters with DAMASK simulations

This notebook post-processes a completed run of the demo workflow
`fit_single_crystal_parameters`. That workflow iteratively fits crystal plasticity
hardening parameters to an experimental uniaxial tension stress-strain curve, using a
bounded Levenberg-Marquardt (LM) scheme. Each loop iteration runs one unperturbed
DAMASK simulation with the current *trial* parameters, plus one simulation per fitting
parameter with that parameter perturbed (these approximate the Jacobian).

A trial is *accepted* if its measured error (the mean absolute difference between the
unperturbed simulated stress and the experimental stress) does not exceed that of the
last accepted trial. Otherwise it is *rejected*: the previous Jacobian is reused with
more damping to propose a new, more conservative trial. Rejected iterations are shown
in grey throughout.

The fit task's `fitting_history` output records accepted/rejected status, errors, trial
parameters, and next proposals. Simulation responses are used only to plot stress-strain
curves.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import LogLocator, NullFormatter, ScalarFormatter

import matflow as mf

## Load the workflow

Set `workflow_path` to the directory (or `.zip` archive) of a completed workflow.

In [ ]:
workflow_path = Path("fit_single_crystal_parameters_2026-11-16_224319.zip")
workflow = mf.Workflow(workflow_path)

# How fitting parameters are scaled to [0, 1] in the parameter-evolution plot:
#   "range": the minimum and maximum trial values
#   "bounds": the perturbation bounds, where defined (falls back to "range")
PARAMETER_SCALING = "range"

## Read fitting history and simulation results

The fit task's `fitting_history` output records accepted/rejected status, errors, trial
parameters, and next proposals. Simulation responses are used only to plot stress-strain
curves.

In [ ]:
def to_curve(VE_response):
    """Return x-normal true strain and stress arrays from a volume-element response."""
    vol_data = VE_response["volume_data"]
    strain = np.asarray(vol_data["vol_avg_strain"]["data"])[..., 0, 0]
    stress = np.asarray(vol_data["vol_avg_stress"]["data"])[..., 0, 0]
    return {"true_strain": strain, "true_stress": stress}


def get_output(element_iteration, name):
    try:
        return getattr(element_iteration.outputs, name).value
    except (AttributeError, ValueError):
        return None


sim_task = workflow.tasks.simulate_VE_loading_damask
fit_task = workflow.tasks.fit_single_crystal_parameters
exp_tensile_test = (
    workflow.tasks.read_tensile_test_from_CSV.elements[0]
    .iterations[0]
    .outputs.tensile_test.value
)
fit_iteration = fit_task.elements[0].iterations[-1]
fitting_history = get_output(fit_iteration, "fitting_history")
if fitting_history is None:
    raise ValueError(
        "This workflow has no fitting_history output. Rerun it with the updated "
        "fit_single_crystal_parameters task schema and script."
    )
if not fitting_history:
    raise ValueError("The workflow fitting_history output is empty.")

fit_inputs = fit_iteration.inputs
try:
    error_tolerance = fit_inputs.fitting_error_tolerance.value
except (AttributeError, ValueError):
    error_tolerance = None

sim_elements = sim_task.elements
first_params = [
    elem.iterations[0].inputs.single_crystal_parameters.value for elem in sim_elements
]
null_idx = next(idx for idx, params in enumerate(first_params) if not params.perturbations)
perturbed_idx = [idx for idx in range(len(sim_elements)) if idx != null_idx]

fitting_param_specs = []
for idx in perturbed_idx:
    perturbation = first_params[idx].perturbations[0]
    fitting_param_specs.append(
        {
            "name": "__".join(str(i) for i in perturbation["path"]),
            "path": perturbation["path"],
            "bounds": perturbation.get("bounds"),
        }
    )
param_names = [spec["name"] for spec in fitting_param_specs]

if len(sim_elements) != len(param_names) + 1:
    raise ValueError(
        "Expected one unperturbed simulation plus one simulation per fitting parameter."
    )
if any(len(elem.iterations) < len(fitting_history) for elem in sim_elements):
    raise ValueError("The workflow has fewer simulation iterations than fit history records.")

iterations = []
for history_record in fitting_history:
    loop_idx = history_record["iteration"] - 1
    elem_iters = [elem.iterations[loop_idx] for elem in sim_elements]
    responses = [get_output(elem_iter, "VE_response") for elem_iter in elem_iters]
    if any(response is None for response in responses):
        raise ValueError(f"Missing simulation response for iteration {loop_idx + 1}.")
    curves = [to_curve(response) for response in responses]
    iterations.append(
        {
            "number": history_record["iteration"],
            "accepted": history_record["accepted"],
            "error": history_record["measured_error"],
            "best_error": history_record["best_measured_error"],
            "trial": history_record["trial_parameters"],
            "best_parameters": history_record["best_parameters"],
            "proposal": history_record["next_trial_parameters"],
            "damping": history_record["next_damping"],
            "tests": [curves[null_idx]] + [curves[idx] for idx in perturbed_idx],
        }
    )

accepted = [it for it in iterations if it["accepted"]]
if not accepted:
    raise ValueError("Fitting history contains no accepted iteration.")
best_it = min(accepted, key=lambda it: it["error"])
final_proposal = iterations[-1]["proposal"]
is_converged = get_output(fit_iteration, "is_converged")

summary = pd.DataFrame(
    [
        {
            "iteration": it["number"],
            "status": "accepted" if it["accepted"] else "rejected",
            "MAE / MPa": it["error"] / 1e6,
            "best MAE / MPa": it["best_error"] / 1e6,
            "next damping": it["damping"],
            **{f"{name} / MPa": it["trial"][name] / 1e6 for name in param_names},
        }
        for it in iterations
    ]
).set_index("iteration")

print(f"Fitting parameters: {param_names}")
print(f"Recorded iterations: {len(iterations)}")
print(f"Best iteration: {best_it['number']} (MAE = {best_it['error'] / 1e6:.3f} MPa)")
if error_tolerance is not None:
    print(f"Acceptable MAE: {error_tolerance / 1e6:g} MPa")
if is_converged is not None:
    print(f"is_converged (workflow output): {is_converged}")
summary.style.format(precision=3)

## Unperturbed stress-strain curves

The unperturbed simulation from each iteration is compared with the experimental
curve. Accepted iterations are coloured from early (dark) to late (light), rejected
iterations are dashed grey, and the best iteration is drawn thicker.

In [ ]:
exp_stress = np.array(exp_tensile_test["true_stress"], dtype=float)
exp_strain = np.array(exp_tensile_test["true_strain"], dtype=float)
exp_valid = np.isfinite(exp_stress) & np.isfinite(exp_strain)
exp_stress, exp_strain = exp_stress[exp_valid], exp_strain[exp_valid]

REJECTED_COLOUR = "0.6"
cmap = plt.get_cmap("viridis")


def iteration_colour(it):
    if not it["accepted"]:
        return REJECTED_COLOUR
    return cmap(0.9 * (it["number"] - 1) / max(len(iterations) - 1, 1))


fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(exp_strain, exp_stress / 1e6, color="k", linestyle=":", linewidth=2, label="Experiment")
for it in iterations:
    test = it["tests"][0]
    is_best = it is best_it
    ax.plot(
        test["true_strain"],
        test["true_stress"] / 1e6,
        color=iteration_colour(it),
        linestyle="-" if it["accepted"] else "--",
        linewidth=2.5 if is_best else 1.2,
        label=(
            f"Iteration {it['number']}"
            + ("" if it["accepted"] else " (rejected)")
            + (" (best)" if is_best else "")
        ),
    )
ax.set_xlim(0, 1.05 * exp_strain.max())
ax.set_xlabel("True strain")
ax.set_ylabel("True stress / MPa")
ax.set_title("Unperturbed simulations vs experiment")
ax.legend(fontsize="small")
fig.tight_layout()

## Perturbed and unperturbed curves at each iteration

Each panel shows one iteration's unperturbed simulation and the simulations with each
fitting parameter perturbed. Rejected iterations have a grey background.

In [ ]:
num_cols = 3
num_rows = int(np.ceil(len(iterations) / num_cols))
fig, axes = plt.subplots(
    num_rows,
    num_cols,
    figsize=(4 * num_cols, 3.2 * num_rows),
    sharex=True,
    sharey=True,
    squeeze=False,
)
for ax, it in zip(axes.flat, iterations):
    ax.plot(exp_strain, exp_stress / 1e6, color="k", linestyle=":", linewidth=2, label="Experiment")
    for sim_idx, test in enumerate(it["tests"]):
        ax.plot(
            test["true_strain"],
            test["true_stress"] / 1e6,
            color=f"C{sim_idx}",
            linewidth=2 if sim_idx == 0 else 1,
            zorder=3 if sim_idx == 0 else 2,
            label="Unperturbed" if sim_idx == 0 else f"Perturbed {param_names[sim_idx - 1]}",
        )
    status = "accepted" if it["accepted"] else "rejected"
    ax.set_title(f"Iteration {it['number']} ({status})\nMAE = {it['error'] / 1e6:.2f} MPa", fontsize="medium")
    if not it["accepted"]:
        ax.set_facecolor("0.9")
for ax in axes.flat[len(iterations):]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("True strain")
for ax in axes[:, 0]:
    ax.set_ylabel("True stress / MPa")
axes.flat[0].set_xlim(0, 1.05 * exp_strain.max())
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize="small")
fig.tight_layout(rect=(0, 0.05, 1, 1))

## Error convergence and fitting parameter evolution

The upper panel shows the measured error for each trial, the best error so far, and the
acceptable error used by the loop termination condition (if set). The lower panel shows
each fitting parameter scaled to $[0, 1]$. The open diamond at the final x-position is
the next proposal from the last fit iteration; it has **not** been simulated and has no
measured error yet. Rejected trials are marked with grey crosses and shaded backgrounds.

In [ ]:
numbers = np.array([it["number"] for it in iterations])
errors = np.array([it["error"] for it in iterations]) / 1e6
best_errors = np.array([it["best_error"] for it in iterations]) / 1e6
is_accepted = np.array([it["accepted"] for it in iterations])
proposal_number = numbers[-1] + 1


def scale_to_unit(name, values):
    spec = next(spec for spec in fitting_param_specs if spec["name"] == name)
    if (
        PARAMETER_SCALING == "bounds"
        and spec["bounds"] is not None
        and all(value is not None for value in spec["bounds"])
    ):
        low, high = spec["bounds"]
    else:
        low = min(values.min(), final_proposal[name])
        high = max(values.max(), final_proposal[name])
    if high == low:
        return np.full_like(values, 0.5), 0.5, (low, high)
    return (values - low) / (high - low), (final_proposal[name] - low) / (high - low), (low, high)


fig, (ax_err, ax_par) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for ax in (ax_err, ax_par):
    for number in numbers[~is_accepted]:
        ax.axvspan(number - 0.5, number + 0.5, color="0.9", zorder=0)

ax_err.plot(numbers, errors, color="0.4", linewidth=1, zorder=1)
ax_err.scatter(numbers[is_accepted], errors[is_accepted], color="C0", zorder=2, label="Accepted trial")
ax_err.scatter(numbers[~is_accepted], errors[~is_accepted], color="0.6", marker="X", s=60, zorder=2, label="Rejected trial")
ax_err.step(numbers, best_errors, where="post", color="C1", label="Best so far")
if error_tolerance is not None:
    ax_err.axhline(error_tolerance / 1e6, color="C3", linestyle="--", label="Acceptable error")
ax_err.set_yscale("log")
ax_err.yaxis.set_major_locator(LogLocator(subs=(1, 2, 5)))
ax_err.yaxis.set_major_formatter(ScalarFormatter())
ax_err.yaxis.set_minor_formatter(NullFormatter())
ax_err.set_ylabel("Mean absolute stress error / MPa")
ax_err.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.01, 1))

for param_idx, name in enumerate(param_names):
    values = np.array([it["trial"][name] for it in iterations])
    scaled, proposal_scaled, (low, high) = scale_to_unit(name, values)
    colour = f"C{param_idx}"
    ax_par.plot(numbers, scaled, color=colour, label=f"{name} [{low / 1e6:.4g}, {high / 1e6:.4g}] MPa")
    ax_par.scatter(numbers[is_accepted], scaled[is_accepted], color=colour, zorder=2)
    ax_par.scatter(numbers[~is_accepted], scaled[~is_accepted], color="0.6", edgecolor=colour, marker="X", s=60, zorder=2)
    ax_par.plot(
        [numbers[-1], proposal_number],
        [scaled[-1], proposal_scaled],
        color=colour,
        linestyle=":",
        linewidth=1,
    )
    ax_par.scatter(
        proposal_number,
        proposal_scaled,
        color="white",
        edgecolor=colour,
        marker="D",
        s=65,
        zorder=3,
    )
ax_par.set_ylim(-0.05, 1.05)
ax_par.set_ylabel(f"Scaled parameter value ({PARAMETER_SCALING})")
ax_par.set_xlabel("Iteration")
ax_par.set_xticks(np.arange(numbers[0], proposal_number + 1))
ax_par.set_xlim(numbers[0] - 0.25, proposal_number + 0.5)
ax_par.scatter([], [], color="white", edgecolor="0.25", marker="D", label="Next proposal (not simulated)")
ax_par.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()

## Best parameters

The parameters from the accepted iteration with the lowest measured error. These are
the `best_single_crystal_parameters` returned by the fit task.

In [ ]:
best_parameters = pd.Series(
    best_it["best_parameters"], name=f"iteration {best_it['number']}"
) / 1e6
best_parameters.to_frame("best value / MPa")